# Netflix Movies and TV Shows: EDA Report

- **Dataset:** Netflix Movies and TV Shows (Kaggle), 8807 rows and 12 columns
- **Goal:** Clean the data and answer 3 questions about Netflix content with charts

In [1]:
import pandas as pd

df = pd.read_csv("data/netflix_titles.csv")

## 1. First look at the data

In [2]:
df.shape

(8807, 12)

In [3]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


In [5]:
df.isna().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

## 2. Findings from the first look

- The data has 8807 rows and 12 columns. Each row is one Netflix title.
- Movies show duration in minutes and TV shows in seasons. It is text, so I cannot calculate with it directly.
- Only release_year is a number. The date_added column is text, so I will convert it to a date later.
- Director has the most missing values (2634). Country and cast also have many.
- There are no duplicate rows.

## 3. My three questions

1. How many Movies and how many TV Shows are on Netflix? (column: type)
2. Which 10 countries have the most titles? (column: country)
3. How many titles were added to Netflix each year? (column: date_added)

## 4. Cleaning plan

I will clean only what my three questions need.

- **Question 1 (type):** nothing to clean.
- **Question 2 (country):** remove the rows with a missing country. Split cells that have more than one country, so each country is counted separately.
- **Question 3 (date_added):** convert the text to a real date, then take the year. Remove the rows with a missing date.
- **Not cleaned:** director, cast, rating and duration, because my questions do not use them.

In [7]:
df_clean = df.copy()
print(df_clean.shape)

(8807, 12)


## 5. Cleaning

In [8]:
df_clean = df.copy()
print(df.shape)
print(df_clean.shape)

(8807, 12)
(8807, 12)


In [9]:
print(df_clean["type"].value_counts())
print(df_clean["type"].isna().sum())

type
Movie      6131
TV Show    2676
Name: count, dtype: int64
0


In [10]:
print(df_clean["country"].isna().sum())
print(df_clean["country"].head(10))
print(df_clean["country"].value_counts().head(5))

831
0                                        United States
1                                         South Africa
2                                                  NaN
3                                                  NaN
4                                                India
5                                                  NaN
6                                                  NaN
7    United States, Ghana, Burkina Faso, United Kin...
8                                       United Kingdom
9                                        United States
Name: country, dtype: str
country
United States     2818
India              972
United Kingdom     419
Japan              245
South Korea        199
Name: count, dtype: int64


In [11]:
df_countries = df_clean[["show_id", "country"]].dropna()
df_countries["country"] = df_countries["country"].str.split(",")
df_countries = df_countries.explode("country")
df_countries["country"] = df_countries["country"].str.strip()
df_countries = df_countries[df_countries["country"] != ""]
print(df_countries["country"].value_counts().head(10))

country
United States     3690
India             1046
United Kingdom     806
Canada             445
France             393
Japan              318
Spain              232
South Korea        231
Germany            226
Mexico             169
Name: count, dtype: int64


In [12]:
missing = df_clean["country"].isna().sum()
missing_pct = ( round(missing/len(df_clean) * 100 , 1))
print(missing)
print(missing_pct)

831
9.4


Finding: 831 shows (9.4%) have no country listed. They are excluded from Question 2 only.

In [13]:
df_clean = df.copy()
df_clean["date_added"] = df_clean["date_added"].str.strip()
df_clean["date_added"] = pd.to_datetime(df_clean["date_added"], format="%B %d, %Y")
print(df_clean["date_added"].dtype)
print(df_clean["date_added"].isna().sum())
print(df_clean["date_added"].head(3))

datetime64[us]
10
0   2021-09-25
1   2021-09-24
2   2021-09-24
Name: date_added, dtype: datetime64[us]


In [14]:
df_clean["year_added"] = df_clean["date_added"].dt.year
print(df_clean["year_added"].value_counts().sort_index())

year_added
2008.0       2
2009.0       2
2010.0       1
2011.0      13
2012.0       3
2013.0      11
2014.0      24
2015.0      82
2016.0     429
2017.0    1188
2018.0    1649
2019.0    2016
2020.0    1879
2021.0    1498
Name: count, dtype: int64


Cleaning done:
- type: no cleaning needed (2 clean groups, 0 missing).
- country: 831 shows (9.4%) have no country. Multi-country cells were split into a separate table, df_countries.
- date_added: converted from text to datetime. 10 dates are missing.